In [9]:
llm_list = ['gemma2','llama3.1']
for llm_model in llm_list:
  !ollama pull {llm_model}

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling ff1d1fc78170... 100% ▕████████████████▏ 5.4 GB                         
pulling 109037bec39c... 100% ▕████████████████▏  136 B                         
pulling 097a36493f71... 100% ▕████████████████▏ 8.4 KB                         
pulling 2490e7468436... 100% ▕████████████████▏   65 B                         
pulling 10aa81da732e... 100% ▕████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 666 MB/4


pulling 667b0c1932bc...  14% ▕██              ▏ 671 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 671 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 672 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 672 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 673 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 673 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 674 MB/4.9 GB  4.4 MB/s   16m8spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 674 MB/4.9 GB  4.4 MB/s   16m8spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 675 MB/4.9 GB  4.7 MB/s  14m59spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 676 MB/4.9 GB  4.7 MB/s  14m59spulling manifest 
pulling 667b0c1932bc...  14% 

In [11]:
import time
import os
import json
import pandas as pd
from tqdm import tqdm
from src.NewsAnalyzerAPI import NewsArticle
from src.NewsAnalyzerAPI import LLMConnector
from src.NewsAnalyzerAPI import NewsAnalyzer

model = "llama3.1"
# model = "gemma2"


def extract_texts(file_path, data_list):
    # extracts specified component from json object
    def extract_component(data, component):
        annotations = data["fiveWoneH"][component]["annotated"]
        texts = [item.get("text") for item in annotations]
        return "; ".join(text for text in texts if text is not None)

    # loads json object from file
    with open(file_path, "r") as file:
        data = json.load(file)

    # extracts true components from the annotated articles (in a json object)
    text = data["text"]
    what_true = extract_component(data, "what")
    where_true = extract_component(data, "where")
    when_true = extract_component(data, "when")
    who_true = extract_component(data, "who")
    why_true = extract_component(data, "why")
    how_true = extract_component(data, "how")

    analyzer = NewsAnalyzer("http://localhost:11434","api_key", model)
    article = NewsArticle(data.get("title"), data.get("description"), text, data.get("date_publish"), data.get("url"))
    analyzer.process_article(article)

    extracted_components = analyzer.extract_components()
    # what_pred = analyzer.identify_component("What")
    # where_pred = analyzer.identify_component("Where")
    # when_pred = analyzer.identify_component("When")
    # who_pred = analyzer.identify_component("Who")
    # why_pred = analyzer.identify_component("Why")
    # how_pred = analyzer.identify_component("How")

    data_list.append({
        "text": text,
        "what_true": what_true,
        "where_true": where_true,
        "when_true": when_true,
        "who_true": who_true,
        "why_true": why_true,
        "how_true": how_true,
        "what_pred": extracted_components["what_pred"],
        "where_pred": extracted_components["where_pred"],
        "when_pred": extracted_components["when_pred"],
        "who_pred": extracted_components["who_pred"],
        "why_pred": extracted_components["why_pred"],
        "how_pred": extracted_components["how_pred"]
    })

In [ ]:
data_list = []
extract_texts("./data_samples/0e5fa7c0e6252bfeeea5e3840c6cb503f299c19d24331c4ba60c5974.json", data_list)
print(data_list)

In [ ]:
pd.DataFrame(data_list)

In [ ]:
!pip install bert_score

In [ ]:
from bert_score import score

In [ ]:
def evaluate(data_list):
    e = []
    for article in data_list:
        cands = [article[component] for component in article if component.endswith("_pred")]
        refs = [article[component] for component in article if component.endswith("_true")]
        # start = time.time()
        P, R, F1 = score(cands, refs, lang="en")
        # end = time.time()
        print(F1)
        print(f"System level F1 score: {F1.mean():.3f}")
        # print("Tempo: ", end-start)
        e.append([article, cands, refs, P, R, F1, end-start])

    return e

In [ ]:
evaluate(data_list)

In [ ]:
# extracting components for all articles and writing in a spreadsheet
data_list = []
data_folder = './data_samples/'
cnt = 1
for filename in tqdm(os.listdir(data_folder)):
    # print(f"\n{cnt}")
    file_path = os.path.join(data_folder, filename)
    # start = time.time()
    extract_texts(file_path, data_list)
    # end = time.time()
    # print("Tempo: ", end-start)
    cnt += 1
    # if len(data_list) > 5: break
df = pd.DataFrame(data_list)
df.to_excel('news_5w1h_llama3.xlsx', index=False)
df.to_csv('news_5w1h_llama3.csv', index=False, encoding='utf-8')

In [ ]:
df

In [ ]:
# gets articles and their extracted components from the spreadsheets and compares extracted components with the true (annotated) components
df = pd.read_excel("news_5w1h_llama3.xlsx")
df = df.fillna("")
data_list2 = df.to_dict(orient="records")
e = evaluate(data_list2)
df_e = pd.DataFrame(e)
df_e['model'] = 'llama3.1'
df_e.to_pickle("news_5w1h_llama3_avaliacao.pkl")

In [ ]:
df_e

# -------------

In [9]:
import time
import os
import json
import pandas as pd
from bert_score import score
from src.NewsAnalyzerAPI import NewsArticle
from src.NewsAnalyzerAPI import LLMConnector
from src.NewsAnalyzerAPI import NewsAnalyzer
from tqdm import tqdm

In [10]:
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("API_KEY")

In [11]:
import logging
from transformers import logging as transformers_logging

# Configuring log level to suppress unwanted warnings
logging.basicConfig(level=logging.ERROR)
transformers_logging.set_verbosity_error()

In [12]:
con = LLMConnector("http://143.107.183.116:18888/v1", api_key, "llama3.1")
analyzer = NewsAnalyzer(con)

In [13]:
# file_path: path to the file with json object with news data
# data_list: list with news data
def extract_texts(file_path, data_list):
    # extracts specified component from json object
    def extract_component(data, component):
        annotations = data["fiveWoneH"][component]["annotated"]
        texts = [item.get("text") for item in annotations]
        return "; ".join(text for text in texts if text is not None)

    # loads json object from file
    with open(file_path, "r") as file:
        data = json.load(file)

    # print("1")
    text = data["text"]
    what_true = extract_component(data, "what")
    where_true = extract_component(data, "where")
    when_true = extract_component(data, "when")
    who_true = extract_component(data, "who")
    why_true = extract_component(data, "why")
    how_true = extract_component(data, "how")
    # print("1")

    # print(data)
    # print("2")
    article = NewsArticle(data.get("title"), data.get("description"), text, data.get("date_publish"), data.get("url"))
    # print("what")
    what_pred = analyzer.identify_component(article, "what")
    
    # print("where")
    where_pred = analyzer.identify_component(article, "where")
    
    # print("when")
    when_pred = analyzer.identify_component(article, "when")
    
    # print("who")
    who_pred = analyzer.identify_component(article, "who")
    
    # print("why")
    why_pred = analyzer.identify_component(article, "why")
    
    # print("how")
    how_pred = analyzer.identify_component(article, "how")
    # print("3")
    # print("2")

    data_list.append({
        "text": text,
        "what_true": what_true,
        "where_true": where_true,
        "when_true": when_true,
        "who_true": who_true,
        "why_true": why_true,
        "how_true": how_true,
        "what_pred": what_pred,
        "where_pred": where_pred,
        "when_pred": when_pred,
        "who_pred": who_pred,
        "why_pred": why_pred,
        "how_pred": how_pred
    })

def evaluate(data_list):
    e = []
    for article in data_list:
        cands = [article[component] for component in article if component.endswith("_pred")]
        refs = [article[component] for component in article if component.endswith("_true")]
        start = time.time()
        P, R, F1 = score(cands, refs, lang="en")
        end = time.time()
        print(F1)
        print(f"System level F1 score: {F1.mean():.3f}")
        print("Tempo: ", end-start)
        e.append([article, cands, refs, P, R, F1, end-start])

    return e

In [ ]:
data_list = []
extract_texts("./data_samples/0e5fa7c0e6252bfeeea5e3840c6cb503f299c19d24331c4ba60c5974.json", data_list)
print(data_list)

In [ ]:
# evaluation for a single article
evaluate(data_list)

In [ ]:
# extracting components for all articles and writing in a spreadsheet
data_list = []
data_folder = './data_samples/'
cnt = 1
for filename in os.listdir(data_folder):
    print(f"\n{cnt}")
    file_path = os.path.join(data_folder, filename)
    start = time.time()
    extract_texts(file_path, data_list)
    end = time.time()
    print("Tempo: ", end-start)
    cnt += 1
df = pd.DataFrame(data_list)
df.to_excel('news.xlsx', index=False)
df.to_csv('news.csv', index=False, encoding='utf-8')

In [5]:
df = pd.read_excel("news.xlsx")
df = df.fillna("")
data_list2 = df.to_dict(orient="records")

In [ ]:
e = evaluate(data_list2)
df_e = pd.DataFrame(e)
df_e.to_pickle("avaliacao.pkl")

In [ ]:
df_e